# ML-Tabelle aufbauen – Spritpreisprognose
Baut die ML-Master-Tabelle aus allen bereinigten Datenquellen.  
Jede Zelle entspricht einem Pipeline-Schritt mit direkter Ausgabe zur Inspektion.

> **Hinweis:** Zellen müssen von oben nach unten ausgeführt werden (Run All empfohlen).

## 0. Imports & Konfiguration

In [1]:
import os
import time
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

# ── Pfade ─────────────────────────────────────────────────────────────────────
# Variante A – Notebook liegt in notebooks/  → "../data"
# Variante B – Notebook liegt im Repo-Root   → "data"
DATA_DIR = Path("../data")
OUT_DIR  = DATA_DIR / "ml"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_WORKERS = max(1, (os.cpu_count() or 2) - 1)

print(f"CPU-Kerne  : {os.cpu_count()}  (Worker: {N_WORKERS})")
print(f"DATA_DIR   : {DATA_DIR.resolve()}")
print(f"OUT_DIR    : {OUT_DIR.resolve()}")

CPU-Kerne  : 20  (Worker: 19)
DATA_DIR   : /home/rex/2_Dokumente/Büro/DSI - Weiterbildung/Abschlussprojekt/GitHub/data
OUT_DIR    : /home/rex/2_Dokumente/Büro/DSI - Weiterbildung/Abschlussprojekt/GitHub/data/ml


## 1. Quelldaten laden

In [2]:
QUELLEN = {
    "preis":         (DATA_DIR / "tankstellen_preise.parquet",    "parquet"),
    "stations":      (DATA_DIR / "tankstellen_stationen.parquet", "parquet"),
    "wetter":        (DATA_DIR / "wetter_koeln.csv",              "csv"),
    "brent":         (DATA_DIR / "brent_futures_daily.csv",       "csv"),
    "eurusd":        (DATA_DIR / "eur_usd_rate.csv",              "csv"),
    "co2":           (DATA_DIR / "co2_abgabe.csv",                "csv"),
    "energiesteuer": (DATA_DIR / "energiesteuer.csv",             "csv"),
    "externe":       (DATA_DIR / "externe_effekte.csv",           "csv"),
    "schulferien":   (DATA_DIR / "schulferien.csv",               "csv"),
    "feiertage":     (DATA_DIR / "feiertage.csv",                 "csv"),
}

def _laden(item):
    key, (pfad, fmt) = item
    t0 = time.perf_counter()
    df = pd.read_parquet(pfad) if fmt == "parquet" else pd.read_csv(pfad)
    return key, df, time.perf_counter() - t0

daten = {}
t_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=min(len(QUELLEN), N_WORKERS + 2)) as pool:
    futures = {pool.submit(_laden, item): item[0] for item in QUELLEN.items()}
    for fut in as_completed(futures):
        key, df, dauer = fut.result()
        daten[key] = df
        print(f"  ✓ {key:<20} {len(df):>10,} Zeilen   {dauer:.2f}s")

print(f"\nGesamt: {len(daten)} Dateien in {time.perf_counter()-t_start:.2f}s")

  ✓ wetter                      552 Zeilen   0.06s
  ✓ schulferien               1,568 Zeilen   0.05s
  ✓ feiertage                 2,823 Zeilen   0.06s
  ✓ externe                   4,225 Zeilen   0.08s
  ✓ brent                     3,071 Zeilen   0.11s
  ✓ energiesteuer             4,225 Zeilen   0.12s
  ✓ eurusd                    6,968 Zeilen   0.15s
  ✓ co2                       4,748 Zeilen   0.19s
  ✓ stations                     30 Zeilen   0.25s
  ✓ preis                 2,430,846 Zeilen   0.71s

Gesamt: 10 Dateien in 0.72s


### Überblick Rohdaten

In [3]:
for key, df in daten.items():
    print(f"{'─'*60}")
    print(f"  {key}   {df.shape}")
    print(f"  Spalten: {df.columns.tolist()}")
print(f"{'─'*60}")

────────────────────────────────────────────────────────────
  wetter   (552, 6)
  Spalten: ['date', 'temp_avg', 'temp_min', 'temp_max', 'niederschlag_mm', 'sonnenstunden']
────────────────────────────────────────────────────────────
  schulferien   (1568, 5)
  Spalten: ['datum_start', 'datum_ende', 'name', 'bundesland_code', 'bundesland_name']
────────────────────────────────────────────────────────────
  feiertage   (2823, 5)
  Spalten: ['datum', 'name', 'bundesland_kuerzel', 'bundesland_name', 'hinweis']
────────────────────────────────────────────────────────────
  externe   (4225, 3)
  Spalten: ['date', 'ist_lockdown', 'ist_niedrigwasser']
────────────────────────────────────────────────────────────
  brent   (3071, 2)
  Spalten: ['period', 'brent_futures_usd']
────────────────────────────────────────────────────────────
  energiesteuer   (4225, 4)
  Spalten: ['date', 'energiesteuer_benzin', 'energiesteuer_diesel', 'ist_tankrabatt']
────────────────────────────────────────────────

## 2. Preisdaten bereinigen

In [4]:
t0 = time.perf_counter()

df_preis = daten["preis"].copy()
df_preis.columns = df_preis.columns.str.lower().str.strip()
df_preis["date"] = pd.to_datetime(df_preis["date"], errors="coerce")
df_preis["station_uuid"] = df_preis["station_uuid"].astype(str)

n_vorher = len(df_preis)
for col in ["diesel", "e5", "e10"]:
    df_preis[col] = pd.to_numeric(df_preis[col], errors="coerce")
    df_preis.loc[(df_preis[col] < 0.5) | (df_preis[col] > 3.0), col] = np.nan

df_preis = df_preis.drop_duplicates()
df_preis = df_preis.sort_values(["station_uuid", "date"]).reset_index(drop=True)

print(f"Zeilen vorher : {n_vorher:,}")
print(f"Zeilen nachher: {len(df_preis):,}  (−{n_vorher - len(df_preis):,})")
print(f"Dauer         : {time.perf_counter()-t0:.2f}s")
print("\nFehlende Werte vor Interpolation:")
display(df_preis[["diesel", "e5", "e10"]].isnull().sum().rename("fehlend").to_frame())

Zeilen vorher : 2,430,846
Zeilen nachher: 2,430,846  (−0)
Dauer         : 3.57s

Fehlende Werte vor Interpolation:


,fehlend
diesel,179
e5,66499
e10,73061


In [5]:
display(df_preis.head(10))

,date,station_uuid,diesel,e5,e10
0,2014-06-08 09:50:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.3390,1.5390,1.4990
1,2014-06-09 00:02:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.4290,1.6290,1.5890
2,2014-06-09 02:38:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.3890,1.5890,1.5490
3,2014-06-09 04:26:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.3590,1.5590,1.5190
4,2014-06-09 05:54:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.3390,1.5390,1.4990
5,2014-06-09 17:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.4290,1.6290,1.5890
6,2014-06-10 01:50:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.3690,1.5690,1.5290
7,2014-06-10 05:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.3490,1.5490,1.5090
8,2014-06-10 09:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.3290,1.5290,1.4890
9,2014-06-10 10:38:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,1.3090,1.5190,1.4790


## 3. Interpolation je Tankstelle

In [6]:
t0 = time.perf_counter()

for col in ["diesel", "e5", "e10"]:
    df_preis[col] = (
        df_preis.groupby("station_uuid")[col]
        .transform(lambda s: s.interpolate(method="linear", limit_direction="both"))
    )

print(f"Fertig in {time.perf_counter()-t0:.2f}s")
print("\nFehlende Werte nach Interpolation:")
display(df_preis[["diesel", "e5", "e10"]].isnull().sum().rename("fehlend").to_frame())

Fertig in 1.25s

Fehlende Werte nach Interpolation:


,fehlend
diesel,0
e5,61645
e10,72827


## 4. Wide → Long + Zeit-Features

In [7]:
t0 = time.perf_counter()

df_preis_long = df_preis.melt(
    id_vars=["date", "station_uuid"],
    value_vars=["diesel", "e5", "e10"],
    var_name="kraftstoff",
    value_name="preis",
)

dt = df_preis_long["date"].dt
df_preis_long["jahr"]           = dt.year.astype("int16")
df_preis_long["quartal"]        = dt.quarter.astype("int8")
df_preis_long["monat"]          = dt.month.astype("int8")
df_preis_long["tag"]            = dt.day.astype("int8")
df_preis_long["stunde"]         = dt.hour.astype("int8")
df_preis_long["wochentag"]      = dt.weekday.astype("int8")
df_preis_long["ist_wochenende"] = df_preis_long["wochentag"].isin([5, 6])
df_preis_long["date_day"]       = dt.floor("D")
df_preis_long["kraftstoff"]     = df_preis_long["kraftstoff"].astype("category")

df_preis_long = df_preis_long.drop_duplicates(subset=["station_uuid", "date", "kraftstoff"])

print(f"Shape: {df_preis_long.shape}   ({time.perf_counter()-t0:.2f}s)")

Shape: (7292538, 12)   (8.25s)


In [8]:
display(df_preis_long.head(12))

,date,station_uuid,kraftstoff,preis,jahr,quartal,monat,tag,stunde,wochentag,ist_wochenende,date_day
0,2014-06-08 09:50:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3390,2014,2,6,8,9,6,True,2014-06-08
1,2014-06-09 00:02:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.4290,2014,2,6,9,0,0,False,2014-06-09
2,2014-06-09 02:38:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3890,2014,2,6,9,2,0,False,2014-06-09
3,2014-06-09 04:26:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3590,2014,2,6,9,4,0,False,2014-06-09
4,2014-06-09 05:54:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3390,2014,2,6,9,5,0,False,2014-06-09
5,2014-06-09 17:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.4290,2014,2,6,9,17,0,False,2014-06-09
6,2014-06-10 01:50:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3690,2014,2,6,10,1,1,False,2014-06-10
7,2014-06-10 05:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3490,2014,2,6,10,5,1,False,2014-06-10
8,2014-06-10 09:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3290,2014,2,6,10,9,1,False,2014-06-10
9,2014-06-10 10:38:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3090,2014,2,6,10,10,1,False,2014-06-10


In [9]:
print("Preisstatistik je Kraftstoff:")
display(df_preis_long.groupby("kraftstoff")["preis"].describe().round(4))

Preisstatistik je Kraftstoff:


,count,mean,std,min,25%,50%,75%,max
kraftstoff,,,,,,,,
diesel,2430846.0000,1.4800,0.2825,0.8590,1.2390,1.5390,1.6590,2.4490
e10,2358019.0000,1.5775,0.2129,1.0590,1.3890,1.6190,1.7290,2.2990
e5,2369201.0000,1.6231,0.2257,0.9190,1.4190,1.6790,1.7890,2.3590


## 5. Stationsdaten bereinigen

In [10]:
df_st = daten["stations"].copy()
df_st.columns = df_st.columns.str.lower().str.strip()

if "uuid" in df_st.columns and "station_uuid" not in df_st.columns:
    df_st["station_uuid"] = df_st["uuid"].astype(str)
elif "station_uuid" in df_st.columns:
    df_st["station_uuid"] = df_st["station_uuid"].astype(str)

for col in ["latitude", "longitude", "distanz_km"]:
    if col in df_st.columns:
        df_st[col] = pd.to_numeric(df_st[col], errors="coerce")

station_cols = [c for c in [
    "station_uuid", "brand", "latitude", "longitude",
    "distanz_km", "post_code", "stadt", "street",
] if c in df_st.columns]
df_stations_slim = df_st[station_cols].drop_duplicates(subset=["station_uuid"])

print(f"{len(df_stations_slim)} Tankstellen")
display(df_stations_slim)

30 Tankstellen


,station_uuid,brand,latitude,longitude,distanz_km,post_code,stadt,street
0,005056ba-7cb6-1ed2-bceb-88651ca7cd30,STAR,50.8882,6.8902,4.3700,50354,koeln,Krankenhausstraße
1,005056ba-7cb6-1ed2-bceb-a51b92434d41,STAR,50.9133,6.8279,1.8711,50226,koeln,Koelner Straße
2,0d04596e-aa4b-44d9-9059-f55beddb0e0b,AVEX,50.8974,6.8210,3.3115,50226,koeln,Gleueler Str.
3,1d79a70e-806c-4eaa-8d8f-5075cff9e67e,Markant,50.9250,6.8953,3.0528,50935,koeln,Dürener Str.
4,1fb2708d-46b8-4cc2-b7c9-d19049ca79c4,Shell,50.9453,6.8314,3.2282,50859,koeln,BRAUWEILER STR. 30-32
5,236b1bde-bf3b-42b5-b8f5-3020286881bc,AVEX,50.9112,6.8301,1.8286,50226,koeln,Bonnstr.
6,24289d2b-3c1c-449d-8f22-b9d52dc37b9e,Westfalen Tankstelle,50.9473,6.8439,3.1477,50859,koeln,Kölner Str.
7,494c4782-ea6e-49bc-90c3-e64dd38b005a,Globus SB Warenhaus,50.9215,6.8533,0.2182,50858,koeln,Max-Planck Straße
8,51d4b453-a095-1aa0-e100-80009459e03a,JET,50.8834,6.8944,4.9732,50354,koeln,LUXEMBURGER STR. 259
9,51d4b50e-a095-1aa0-e100-80009459e03a,JET,50.9450,6.9077,4.7846,50825,koeln,WIDDERSDORFER STR. 165


## 6. Externe Features vorbereiten

In [11]:
def tages_join(df, date_col, keep_cols, renames=None):
    """Normalisiert ein DataFrame auf Tagesebene für den Left-Join."""
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    if renames:
        df = df.rename(columns=renames)
    df["date_day"] = pd.to_datetime(df[date_col], errors="coerce").dt.floor("D")
    df = df.sort_values("date_day").drop_duplicates(subset=["date_day"], keep="last")
    cols = ["date_day"] + [c for c in keep_cols if c in df.columns]
    return df[cols].reset_index(drop=True)

def _prep_wetter(df):
    return tages_join(df, "date",
        ["temp_avg", "temp_min", "temp_max", "niederschlag_mm", "sonnenstunden"])

def _prep_brent(df):
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    renames = {}
    if "period" in df.columns:            renames["period"] = "date"
    if "brent_futures_usd" in df.columns: renames["brent_futures_usd"] = "brent_usd"
    if renames: df = df.rename(columns=renames)
    df["brent_usd"] = pd.to_numeric(df["brent_usd"], errors="coerce")
    return tages_join(df, "date", ["brent_usd"])

def _prep_eurusd(df):
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    if "period" in df.columns: df = df.rename(columns={"period": "date"})
    df["eur_usd"] = pd.to_numeric(df["eur_usd"], errors="coerce")
    return tages_join(df, "date", ["eur_usd"])

def _prep_co2(df):
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    for col in df.columns:
        if "co2" in col: df[col] = pd.to_numeric(df[col], errors="coerce")
    return tages_join(df, "date", ["co2_preis_eur_t"])

def _prep_energiesteuer(df):
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    for col in ["energiesteuer_benzin", "energiesteuer_diesel"]:
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors="coerce")
    if "ist_tankrabatt" in df.columns:
        df["ist_tankrabatt"] = df["ist_tankrabatt"].astype(bool)
    return tages_join(df, "date",
        ["energiesteuer_benzin", "energiesteuer_diesel", "ist_tankrabatt"])

def _prep_externe(df):
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()
    for col in ["ist_lockdown", "ist_niedrigwasser"]:
        if col in df.columns: df[col] = df[col].astype(bool)
    return tages_join(df, "date", ["ist_lockdown", "ist_niedrigwasser"])

prep_tasks = [
    ("wetter",        _prep_wetter,        daten["wetter"]),
    ("brent",         _prep_brent,         daten["brent"]),
    ("eurusd",        _prep_eurusd,        daten["eurusd"]),
    ("co2",           _prep_co2,           daten["co2"]),
    ("energiesteuer", _prep_energiesteuer, daten["energiesteuer"]),
    ("externe",       _prep_externe,       daten["externe"]),
]

t0 = time.perf_counter()
join_dfs = {}
with ThreadPoolExecutor(max_workers=len(prep_tasks)) as pool:
    future_map = {pool.submit(fn, df): name for name, fn, df in prep_tasks}
    for fut in as_completed(future_map):
        name = future_map[fut]
        join_dfs[name] = fut.result()
        print(f"  ✓ {name:<20} {len(join_dfs[name]):>5} Tage")

print(f"\nFertig in {time.perf_counter()-t0:.2f}s")

  ✓ wetter                 552 Tage
  ✓ externe               4225 Tage
  ✓ eurusd                6968 Tage
  ✓ brent                 3071 Tage
  ✓ energiesteuer         4225 Tage
  ✓ co2                   4748 Tage

Fertig in 0.17s


### Stichproben externe Features

In [12]:
for name, df in join_dfs.items():
    print(f"\n── {name}")
    display(df.head(3))


── wetter


,date_day,temp_avg,temp_min,temp_max,niederschlag_mm,sonnenstunden
0,2024-09-13,10.0000,4.2000,15.2000,1.5000,3.3000
1,2024-09-14,9.9000,5.0000,17.0000,0.0000,6.0000
2,2024-09-15,12.1000,4.2000,18.5000,0.0000,8.5000



── externe


,date_day,ist_lockdown,ist_niedrigwasser
0,2014-06-08,False,False
1,2014-06-09,False,False
2,2014-06-10,False,False



── eurusd


...

,date_day,brent_usd
0,2014-01-02,107.7800
1,2014-01-03,106.8900
2,2014-01-06,106.7300



── energiesteuer


,date_day,energiesteuer_benzin,energiesteuer_diesel,ist_tankrabatt
0,2014-06-08,65.4500,47.0400,False
1,2014-06-09,65.4500,47.0400,False
2,2014-06-10,65.4500,47.0400,False



── co2


,date_day,co2_preis_eur_t
0,2014-01-01,0.0000
1,2014-01-02,0.0000
2,2014-01-03,0.0000


## 7. ML-Tabelle zusammenführen

In [13]:
t0 = time.perf_counter()

df_ml = (
    df_preis_long
    .merge(join_dfs["wetter"],        on="date_day", how="left")
    .merge(join_dfs["brent"],         on="date_day", how="left")
    .merge(join_dfs["eurusd"],        on="date_day", how="left")
    .merge(join_dfs["co2"],           on="date_day", how="left")
    .merge(join_dfs["energiesteuer"], on="date_day", how="left")
    .merge(join_dfs["externe"],       on="date_day", how="left")
)

print(f"Shape: {df_ml.shape}   ({time.perf_counter()-t0:.2f}s)")
display(df_ml.head(10))

Shape: (7292538, 25)   (45.37s)


,date,station_uuid,kraftstoff,preis,jahr,quartal,monat,tag,stunde,wochentag,ist_wochenende,date_day,temp_avg,temp_min,temp_max,niederschlag_mm,sonnenstunden,brent_usd,eur_usd,co2_preis_eur_t,energiesteuer_benzin,energiesteuer_diesel,ist_tankrabatt,ist_lockdown,ist_niedrigwasser
0,2014-06-08 09:50:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3390,2014,2,6,8,9,6,True,2014-06-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0000,65.4500,47.0400,False,False,False
1,2014-06-09 00:02:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.4290,2014,2,6,9,0,0,False,2014-06-09,NaN,NaN,NaN,NaN,NaN,109.9900,1.3608,0.0000,65.4500,47.0400,False,False,False
2,2014-06-09 02:38:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3890,2014,2,6,9,2,0,False,2014-06-09,NaN,NaN,NaN,NaN,NaN,109.9900,1.3608,0.0000,65.4500,47.0400,False,False,False
3,2014-06-09 04:26:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3590,2014,2,6,9,4,0,False,2014-06-09,NaN,NaN,NaN,NaN,NaN,109.9900,1.3608,0.0000,65.4500,47.0400,False,False,False
4,2014-06-09 05:54:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3390,2014,2,6,9,5,0,False,2014-06-09,NaN,NaN,NaN,NaN,NaN,109.9900,1.3608,0.0000,65.4500,47.0400,False,False,False
5,2014-06-09 17:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.4290,2014,2,6,9,17,0,False,2014-06-09,NaN,NaN,NaN,NaN,NaN,109.9900,1.3608,0.0000,65.4500,47.0400,False,False,False
6,2014-06-10 01:50:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3690,2014,2,6,10,1,1,False,2014-06-10,NaN,NaN,NaN,NaN,NaN,109.5200,1.3547,0.0000,65.4500,47.0400,False,False,False
7,2014-06-10 05:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3490,2014,2,6,10,5,1,False,2014-06-10,NaN,NaN,NaN,NaN,NaN,109.5200,1.3547,0.0000,65.4500,47.0400,False,False,False
8,2014-06-10 09:06:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3290,2014,2,6,10,9,1,False,2014-06-10,NaN,NaN,NaN,NaN,NaN,109.5200,1.3547,0.0000,65.4500,47.0400,False,False,False
9,2014-06-10 10:38:01,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3090,2014,2,6,10,10,1,False,2014-06-10,NaN,NaN,NaN,NaN,NaN,109.5200,1.3547,0.0000,65.4500,47.0400,False,False,False


## 8. Schulferien & Feiertage

In [14]:
# NRW-Schulferien: Datumsbereiche → Set für O(1)-Lookup
df_sf = daten["schulferien"].copy()
df_sf.columns = df_sf.columns.str.lower().str.strip()
df_sf["datum_start"] = pd.to_datetime(df_sf["datum_start"], errors="coerce")
df_sf["datum_ende"]  = pd.to_datetime(df_sf["datum_ende"],  errors="coerce")
nrw_ferien = df_sf[df_sf["bundesland_code"].str.upper().str.contains("NW|NRW", na=False)]

ferien_tage = set()
for _, row in nrw_ferien.iterrows():
    if pd.notna(row["datum_start"]) and pd.notna(row["datum_ende"]):
        ferien_tage.update(pd.date_range(row["datum_start"], row["datum_ende"], freq="D"))

df_ml["is_schulferien"] = df_ml["date_day"].isin(ferien_tage)
print(f"NRW-Ferientage : {len(ferien_tage)}")

# NRW-Feiertage
df_ft = daten["feiertage"].copy()
df_ft.columns = df_ft.columns.str.lower().str.strip()
df_ft["datum"] = pd.to_datetime(df_ft["datum"], errors="coerce")
nrw_feiertage_set = set(
    df_ft[
        df_ft["bundesland_kuerzel"].str.upper().str.contains("NW|NRW|DE", na=False)
    ]["datum"].dropna().dt.floor("D")
)
df_ml["is_feiertag"] = df_ml["date_day"].isin(nrw_feiertage_set)
print(f"NRW-Feiertage  : {len(nrw_feiertage_set)}")

NRW-Ferientage : 1313
NRW-Feiertage  : 166


## 9. Fehlende Werte auffüllen & Dtypes optimieren

In [15]:
tages_features = [
    "temp_avg", "temp_min", "temp_max", "niederschlag_mm", "sonnenstunden",
    "brent_usd", "eur_usd", "co2_preis_eur_t",
    "energiesteuer_benzin", "energiesteuer_diesel",
    "ist_tankrabatt", "ist_lockdown", "ist_niedrigwasser",
]
df_ml = df_ml.sort_values("date_day").reset_index(drop=True)

for col in tages_features:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].ffill().bfill()

for col in ["ist_tankrabatt", "ist_lockdown", "ist_niedrigwasser",
            "ist_wochenende", "is_schulferien", "is_feiertag"]:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].astype(bool)

# float64 → float32
for col in ["preis", "brent_usd", "eur_usd", "co2_preis_eur_t",
            "energiesteuer_benzin", "energiesteuer_diesel",
            "temp_avg", "temp_min", "temp_max", "niederschlag_mm", "sonnenstunden"]:
    if col in df_ml.columns:
        df_ml[col] = df_ml[col].astype("float32")

ram_mb = df_ml.memory_usage(deep=True).sum() / 1_048_576
print(f"RAM: {ram_mb:.0f} MB")

missing = df_ml.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if missing.empty:
    print("Keine fehlenden Werte ✓")
else:
    display(missing.to_frame("fehlend"))

RAM: 1106 MB


,fehlend
preis,134472


## 10. Analyse-Dataset aufbauen (+ Stationsdaten)

In [16]:
analysis_col_order = [
    "date", "date_day", "station_uuid",
    "kraftstoff", "preis",
    "jahr", "quartal", "monat", "tag", "stunde", "wochentag",
    "ist_wochenende", "is_schulferien", "is_feiertag",
    "brent_usd", "eur_usd", "co2_preis_eur_t",
    "energiesteuer_benzin", "energiesteuer_diesel", "ist_tankrabatt",
    "temp_avg", "temp_min", "temp_max", "niederschlag_mm", "sonnenstunden",
    "ist_lockdown", "ist_niedrigwasser",
    "brand", "post_code", "stadt", "street",
    "latitude", "longitude", "distanz_km",
]

# Merge nur die benötigten Station-Spalten, kein .copy() → spart RAM
station_extra = [c for c in ["brand", "post_code", "stadt", "street", "latitude", "longitude", "distanz_km"]
                 if c in df_stations_slim.columns]
df_analysis = df_ml.merge(df_stations_slim[["station_uuid"] + station_extra],
                           on="station_uuid", how="left")
analysis_col_order = [c for c in analysis_col_order if c in df_analysis.columns]
df_analysis = df_analysis[analysis_col_order]

print(f"Shape: {df_analysis.shape}")
display(df_analysis.head(8))

Shape: (7292538, 34)


,date,date_day,station_uuid,kraftstoff,preis,jahr,quartal,monat,tag,stunde,wochentag,ist_wochenende,is_schulferien,is_feiertag,brent_usd,eur_usd,co2_preis_eur_t,energiesteuer_benzin,energiesteuer_diesel,ist_tankrabatt,temp_avg,temp_min,temp_max,niederschlag_mm,sonnenstunden,ist_lockdown,ist_niedrigwasser,brand,post_code,stadt,street,latitude,longitude,distanz_km
0,2014-06-08 09:50:01,2014-06-08,005056ba-7cb6-1ed2-bceb-88651ca7cd30,diesel,1.3390,2014,2,6,8,9,6,True,False,False,109.9900,1.3608,0.0000,65.4500,47.0400,False,10.0000,4.2000,15.2000,1.5000,3.3000,False,False,STAR,50354,koeln,Krankenhausstraße,50.8882,6.8902,4.3700
1,2014-06-08 09:50:01,2014-06-08,51d4b453-a095-1aa0-e100-80009459e03a,e5,1.5290,2014,2,6,8,9,6,True,False,False,109.9900,1.3608,0.0000,65.4500,47.0400,False,10.0000,4.2000,15.2000,1.5000,3.3000,False,False,JET,50354,koeln,LUXEMBURGER STR. 259,50.8834,6.8944,4.9732
2,2014-06-08 17:02:01,2014-06-08,7e2d0df2-59cc-4fb1-896f-0a6a090236b6,e10,1.6190,2014,2,6,8,17,6,True,False,False,109.9900,1.3608,0.0000,65.4500,47.0400,False,10.0000,4.2000,15.2000,1.5000,3.3000,False,False,SB,50858,koeln,Aachener Str. 1035,50.9379,6.8652,2.2231
3,2014-06-08 09:50:01,2014-06-08,7e2d0df2-59cc-4fb1-896f-0a6a090236b6,e10,1.5390,2014,2,6,8,9,6,True,False,False,109.9900,1.3608,0.0000,65.4500,47.0400,False,10.0000,4.2000,15.2000,1.5000,3.3000,False,False,SB,50858,koeln,Aachener Str. 1035,50.9379,6.8652,2.2231
4,2014-06-08 09:50:01,2014-06-08,0d04596e-aa4b-44d9-9059-f55beddb0e0b,diesel,1.3190,2014,2,6,8,9,6,True,False,False,109.9900,1.3608,0.0000,65.4500,47.0400,False,10.0000,4.2000,15.2000,1.5000,3.3000,False,False,AVEX,50226,koeln,Gleueler Str.,50.8974,6.8210,3.3115
5,2014-06-08 09:50:01,2014-06-08,0d04596e-aa4b-44d9-9059-f55beddb0e0b,e10,1.4990,2014,2,6,8,9,6,True,False,False,109.9900,1.3608,0.0000,65.4500,47.0400,False,10.0000,4.2000,15.2000,1.5000,3.3000,False,False,AVEX,50226,koeln,Gleueler Str.,50.8974,6.8210,3.3115
6,2014-06-08 09:50:01,2014-06-08,833c5106-cf84-4da5-b427-71839081b391,diesel,1.3190,2014,2,6,8,9,6,True,False,False,109.9900,1.3608,0.0000,65.4500,47.0400,False,10.0000,4.2000,15.2000,1.5000,3.3000,False,False,AVEX,50226,koeln,Bonnstr.,50.9112,6.8301,1.8286
7,2014-06-08 09:50:01,2014-06-08,51d4b50e-a095-1aa0-e100-80009459e03a,e10,1.4990,2014,2,6,8,9,6,True,False,False,109.9900,1.3608,0.0000,65.4500,47.0400,False,10.0000,4.2000,15.2000,1.5000,3.3000,False,False,JET,50825,koeln,WIDDERSDORFER STR. 165,50.9450,6.9077,4.7846


## 11. Qualitäts-Check

In [17]:
print(f"Zeitraum : {df_ml['date'].min().date()} → {df_ml['date'].max().date()}")
print(f"Zeilen   : {len(df_ml):,}")
print(f"Spalten  : {df_ml.shape[1]}")
print()
print("Kraftstoff-Verteilung:")
display(df_ml["kraftstoff"].value_counts().to_frame())

Zeitraum : 2014-06-08 → 2026-03-19
Zeilen   : 7,292,538
Spalten  : 27

Kraftstoff-Verteilung:


,count
kraftstoff,
diesel,2430846
e10,2430846
e5,2430846


In [18]:
print("Preisstatistik je Kraftstoff:")
display(df_ml.groupby("kraftstoff")["preis"].describe().round(4))

Preisstatistik je Kraftstoff:


,count,mean,std,min,25%,50%,75%,max
kraftstoff,,,,,,,,
diesel,2430846.0000,1.4800,0.2821,0.8590,1.2390,1.5390,1.6590,2.4490
e10,2358019.0000,1.5775,0.2126,1.0590,1.3890,1.6190,1.7290,2.2990
e5,2369201.0000,1.6231,0.2260,0.9190,1.4190,1.6790,1.7890,2.3590


In [19]:
print("Alle numerischen Features:")
display(df_ml.describe(include="all").T)

Alle numerischen Features:


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
date,7292538,NaN,NaN,NaN,2021-12-26 03:53:08.112605184,2014-06-08 09:50:01,2019-11-03 12:45:04,2022-06-14 18:23:11,2024-08-31 09:30:57,2026-03-19 23:17:55,NaN
station_uuid,7292538,30,1d79a70e-806c-4eaa-8d8f-5075cff9e67e,339063,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kraftstoff,7292538,3,diesel,2430846,NaN,NaN,NaN,NaN,NaN,NaN,NaN
preis,7158066.0000,NaN,NaN,NaN,1.5595,0.8590,1.3590,1.5990,1.7290,2.4490,0.2571
jahr,7292538.0000,NaN,NaN,NaN,2021.4743,2014.0000,2019.0000,2022.0000,2024.0000,2026.0000,3.0364
quartal,7292538.0000,NaN,NaN,NaN,2.5455,1.0000,2.0000,3.0000,4.0000,4.0000,1.1189
monat,7292538.0000,NaN,NaN,NaN,6.6402,1.0000,4.0000,7.0000,10.0000,12.0000,3.4651
tag,7292538.0000,NaN,NaN,NaN,15.7267,1.0000,8.0000,16.0000,23.0000,31.0000,8.7826
stunde,7292538.0000,NaN,NaN,NaN,13.3862,0.0000,10.0000,13.0000,17.0000,23.0000,4.7074
wochentag,7292538.0000,NaN,NaN,NaN,2.9435,0.0000,1.0000,3.0000,5.0000,6.0000,1.9873


In [20]:
bool_feats = [c for c in df_ml.columns if df_ml[c].dtype == bool]
print("Bool-Features (Anteil True):")
display(df_ml[bool_feats].mean().rename("anteil_true").to_frame().round(4))

Bool-Features (Anteil True):


,anteil_true
ist_wochenende,0.2723
ist_tankrabatt,0.0254
ist_lockdown,0.0218
ist_niedrigwasser,0.0269
is_schulferien,0.2394
is_feiertag,0.0277


In [21]:
print("Spalten & Dtypes:")
display(df_ml.dtypes.rename("dtype").to_frame())

Spalten & Dtypes:


,dtype
date,datetime64[ns]
station_uuid,object
kraftstoff,category
preis,float32
jahr,int16
quartal,int8
monat,int8
tag,int8
stunde,int8
wochentag,int8


## 12. ML-Export (Wide-Format)
Laut Spec: `timestamp`, `station_uuid`, `e5`, `e10`, `diesel` + alle Features als eigene Spalten.

In [ ]:
t0 = time.perf_counter()

index_cols = [c for c in [
    "date", "station_uuid",
    "brent_usd", "eur_usd",
    "is_feiertag", "is_schulferien",
    "temp_avg", "niederschlag_mm", "sonnenstunden",
    "ist_tankrabatt", "co2_preis_eur_t",
    "ist_lockdown", "ist_niedrigwasser",
] if c in df_ml.columns]

df_export = (
    df_ml[index_cols + ["kraftstoff", "preis"]]
    .drop_duplicates(subset=index_cols + ["kraftstoff"])
    .set_index(index_cols + ["kraftstoff"])["preis"]
    .unstack("kraftstoff")
    .reset_index()
)
df_export.columns.name = None

df_export = df_export.rename(columns={
    "date":        "timestamp",
    "brent_usd":   "brent_futures_usd",
})

col_order = [
    "timestamp", "station_uuid",
    "e5", "e10", "diesel",
    "brent_futures_usd", "eur_usd",
    "is_feiertag", "is_schulferien",
    "temp_avg", "niederschlag_mm", "sonnenstunden",
    "ist_tankrabatt", "co2_preis_eur_t",
    "ist_lockdown", "ist_niedrigwasser",
]
col_order = [c for c in col_order if c in df_export.columns]
df_export = df_export[col_order]

print(f"Shape: {df_export.shape}   ({time.perf_counter()-t0:.2f}s)")
display(df_export.head(10))

## 13. Speichern

In [ ]:
t0 = time.perf_counter()

paths = {
    "ml_master_dataset.parquet": df_ml,
    "analysis_dataset.parquet":  df_analysis,
    "ml_export.parquet":         df_export,
}

def _speichern(item):
    name, df = item
    pfad = OUT_DIR / name
    df.to_parquet(pfad, index=False, engine="pyarrow", compression="snappy")
    return pfad

with ThreadPoolExecutor(max_workers=3) as pool:
    for pfad in pool.map(_speichern, paths.items()):
        mb = pfad.stat().st_size / 1_048_576
        print(f"  ✓ {pfad}  ({mb:.1f} MB)")

print(f"\nFertig in {time.perf_counter()-t0:.2f}s")

## 14. Ausgabe inspizieren

In [ ]:
display(pd.read_parquet(OUT_DIR / "ml_master_dataset.parquet").head(20))

In [ ]:
display(pd.read_parquet(OUT_DIR / "analysis_dataset.parquet").head(20))

In [ ]:
display(pd.read_parquet(OUT_DIR / "ml_export.parquet").head(20))